# Session 1 — Look at the Data Before You Touch a Model
### Brain MRI Tumour Segmentation · CS Academy Seminar

---

Somewhere in the image below there is a brain tumour.

Today you will **not** build a model. You will look at real MRI scans from **110 patients** with
lower-grade glioma, released by The Cancer Imaging Archive. A board-certified radiologist at Duke
University drew the outline of the tumour on every single slice by hand.

Before you touch machine learning, hold on to one number:

> **Two expert radiologists, given the same scan, only agree on where the tumour is about 84% of the time.**
> *(Buda et al., 2019 — 84% Dice, standard deviation 2%)*

So when your model scores 80%, is that bad? Is 90% suspiciously good? Keep asking that all week.

**Today's goal:** produce a *dataset report*. By the end you should be able to answer, from your own
code, every question in the checklist at the bottom.

⏱ Roughly 2 hours.

In [ ]:
#@title Setup — run this first  { display-mode: "form" }
# Downloads the seminar helper code and the dataset.
REPO_RAW = "https://raw.githubusercontent.com/OTMAN-REPO/brain-mri-seminar/main"  #@param {type:"string"}
DATA_URL = ""  #@param {type:"string"}

import os, urllib.request
if not os.path.exists("seminar.py"):
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/seminar.py", "seminar.py")
        print("Got seminar.py")
    except Exception as e:
        raise SystemExit(f"Could not fetch seminar.py from {REPO_RAW}\n"
                         f"Upload it manually to this Colab session (folder icon on the left).\n{e}")

from seminar import *
import numpy as np, matplotlib.pyplot as plt
images, masks, patient_ids, slice_index = get_data(url=DATA_URL)
print(f"\n{len(images)} slices | {len(np.unique(patient_ids))} patients | image {images.shape[1:]}")
print("device:", DEVICE)

## 1. What am I actually looking at?

Each patient has a stack of 2D **slices** — imagine slicing a loaf of bread and photographing each slice.

Each slice here has **3 channels**, which are three different MRI sequences of the same anatomy:

| channel | sequence | what it shows |
|---|---|---|
| 0 | pre-contrast | baseline anatomy |
| 1 | **FLAIR** | fluid suppressed, so lesions glow bright — this is where the tumour is easiest to see |
| 2 | post-contrast | after injecting contrast dye |

FLAIR stands for *Fluid-Attenuated Inversion Recovery*. It darkens the cerebrospinal fluid so that
abnormal tissue stands out. It is the channel radiologists lean on for this task.

The **mask** is the label: 1 where the radiologist said "tumour", 0 everywhere else.

In [ ]:
# Look at one slice, channel by channel
i = int(np.argmax(masks.reshape(len(masks), -1).sum(1)))   # the slice with the biggest tumour

fig, ax = plt.subplots(1, 5, figsize=(16, 3.4))
for c, name in enumerate(["pre-contrast", "FLAIR", "post-contrast"]):
    ax[c].imshow(images[i][..., c], cmap="gray"); ax[c].set_title(name)
ax[3].imshow(masks[i], cmap="gray"); ax[3].set_title("mask (the label)")
ax[4].imshow(images[i][..., 1], cmap="gray")
ax[4].contour(masks[i], levels=[0.5], colors="lime", linewidths=1.5)
ax[4].set_title("FLAIR + outline")
for a in ax: a.axis("off")
plt.suptitle(f"patient {patient_ids[i]}, slice {slice_index[i]}", y=1.04)
plt.tight_layout(); plt.show()

## 2. Scroll through a whole patient

A tumour is a 3D object. It appears on some slices, grows, then disappears.
Move the slider and watch it come and go.

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown

pats = sorted(np.unique(patient_ids))

def show(patient, slice_no):
    sel = np.where(patient_ids == patient)[0]
    j = sel[min(slice_no, len(sel) - 1)]
    fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.8))
    ax[0].imshow(images[j][..., 1], cmap="gray"); ax[0].set_title("FLAIR")
    ax[1].imshow(images[j][..., 1], cmap="gray")
    if masks[j].any():
        ax[1].contour(masks[j], levels=[0.5], colors="lime", linewidths=1.5)
    px = int(masks[j].sum())
    ax[1].set_title(f"tumour pixels: {px}" if px else "no tumour on this slice")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

interact(show, patient=Dropdown(options=pats, value=pats[0]),
         slice_no=IntSlider(0, 0, 40, 1));

## 3. Your dataset report

From here on, **you** write the code. Each cell has a `TODO`. Do not skip ahead — the questions below
are the ones that will decide how you build your model next session.

### Q1. How big is this dataset, really?

In [ ]:
# TODO: print the number of patients, the number of slices,
#       and the average number of slices per patient.
#
# hints:  np.unique(patient_ids)          -> the distinct patients
#         len(images)                     -> total slices

n_patients = ...
n_slices   = ...
print(f"{n_patients} patients, {n_slices} slices, {n_slices/n_patients:.1f} slices per patient")

### Q2. What fraction of slices actually contain a tumour?

This one matters more than it looks. If you know the answer, you can already guess how a lazy model
might cheat.

In [ ]:
# TODO: compute the fraction of slices where the mask contains at least one tumour pixel.
# hint: masks[k].any() is True if slice k has any tumour.
#       Or vectorised: masks.reshape(len(masks), -1).sum(1) > 0

has_tumour = ...
print(f"{has_tumour.mean():.1%} of slices contain tumour")
print(f"{has_tumour.sum()} with tumour, {(~has_tumour).sum()} without")

### Q3. What fraction of *pixels* are tumour?

Not slices — individual pixels, across the entire dataset.

In [ ]:
# TODO: compute the fraction of all pixels in the dataset labelled as tumour.
tumour_pixel_fraction = ...
print(f"{tumour_pixel_fraction:.3%} of pixels are tumour")
print(f"So {1 - tumour_pixel_fraction:.3%} of pixels are background.")
print("\nSit with that number for a second. We will come back to it next session.")

### Q4. How big are the tumours, and how much do they vary?

In [ ]:
areas = masks.reshape(len(masks), -1).sum(1)
areas_nz = areas[areas > 0]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(areas_nz, bins=50, color="steelblue")
ax[0].set_xlabel("tumour pixels in a slice"); ax[0].set_ylabel("count")
ax[0].set_title("tumour size distribution (tumour slices only)")
ax[1].hist(areas_nz, bins=50, color="indianred"); ax[1].set_yscale("log")
ax[1].set_xlabel("tumour pixels in a slice"); ax[1].set_title("same thing, log scale")
plt.tight_layout(); plt.show()

print(f"smallest tumour: {areas_nz.min()} px | median: {np.median(areas_nz):.0f} px | largest: {areas_nz.max()} px")
print(f"The largest tumour is {areas_nz.max()/areas_nz.min():.0f}x the area of the smallest.")

# TODO: how many tumour slices have fewer than 20 pixels of tumour?
#       Do you think a model will find those? Do you think Dice will be fair to them?

### Q5. Are all patients the same?

The patient IDs look like `TCGA_CS_4941_19960909`. That `CS` / `DU` / `FG` / `HT` part is the
**institution** the scan came from. Different hospitals, different scanners, different image quality.

In [ ]:
# TODO: extract the institution code (the 2nd underscore-separated field) from each patient id,
#       and count how many patients came from each institution.
#
# hint: "TCGA_CS_4941_19960909".split("_")[1]  ->  "CS"

import collections
institutions = ...
print(collections.Counter(institutions))

# Think about: if you train on one hospital and test on another, what might happen?

## 4. The thing you are going to build

Here is a U-Net trained on this exact dataset by the researchers who published it.
It has never seen you, and you have not written a line of model code yet.

Run it, and look at what comes out.

In [ ]:
import torch
from skimage.exposure import rescale_intensity
from skimage.transform import resize

def normalize_volume(vol):
    """Exactly the preprocessing the pretrained model was trained on (Buda et al.)."""
    p10, p99 = np.percentile(vol, 10), np.percentile(vol, 99)
    vol = rescale_intensity(vol, in_range=(p10, p99))
    return (vol - vol.mean(axis=(0,1,2))) / vol.std(axis=(0,1,2))

def prep_for_pretrained(picks, size=256):
    """Normalise each slice within ITS OWN patient volume, then upsample to 256."""
    out = []
    for j in picks:
        sel = np.where(patient_ids == patient_ids[j])[0]
        sl = normalize_volume(images[sel].astype(np.float32))[int(np.where(sel == j)[0][0])]
        if sl.shape[0] != size:
            sl = resize(sl, (size, size, 3), order=2, mode="constant", cval=0,
                        anti_aliasing=False)
        out.append(sl.transpose(2, 0, 1))
    return np.stack(out).astype(np.float32)

def pick_slices(areas, patient_ids, n=6, lo=0.30, hi=0.98):
    """Six tumour slices spread across the size range, one per patient.

    Taking the six LARGEST tumours instead would flatter the model badly -- Dice
    is size-biased -- and they tend to come from only one or two patients.
    """
    ranked = np.where(areas > 0)[0]
    ranked = ranked[np.argsort(areas[ranked])]
    picks, seen = [], set()
    for frac in np.linspace(lo, hi, n):
        start = int(frac * (len(ranked) - 1))
        for i in sorted(range(len(ranked)), key=lambda i: abs(i - start)):
            j = ranked[i]
            if patient_ids[j] not in seen:
                picks.append(j); seen.add(patient_ids[j]); break
    return np.array(picks)

try:
    demo = torch.hub.load("mateuszbuda/brain-segmentation-pytorch", "unet",
                          in_channels=3, out_channels=1, init_features=32,
                          pretrained=True, trust_repo=True).to(DEVICE).eval()

    areas = masks.reshape(len(masks), -1).sum(1)
    picks = pick_slices(areas, patient_ids)

    x = torch.from_numpy(prep_for_pretrained(picks)).to(DEVICE)
    with torch.no_grad():
        prob = demo(x).cpu().numpy()[:, 0]      # NOTE: model already applies sigmoid

    # bring predictions back to our 128x128 grid for overlay and scoring
    H = masks.shape[1]
    pred = np.stack([resize(p, (H, H), order=1, mode="constant",
                            cval=0, anti_aliasing=False) for p in prob]) > 0.5

    fig, ax = plt.subplots(2, 6, figsize=(16, 5.8))
    for k, j in enumerate(picks):
        ax[0,k].imshow(images[j][...,1], cmap="gray")
        ax[0,k].contour(masks[j], levels=[.5], colors="lime", linewidths=1.2)
        ax[0,k].set_title(f"radiologist\n{int(masks[j].sum())} px", fontsize=8)
        ax[1,k].imshow(images[j][...,1], cmap="gray")
        if pred[k].any():
            ax[1,k].contour(pred[k], levels=[.5], colors="red", linewidths=1.2)
        ax[1,k].set_title(f"model · dice {dice_score(pred[k], masks[j]):.2f}", fontsize=8)
    for a in ax.ravel(): a.axis("off")
    plt.suptitle("green = radiologist    red = the machine", y=1.02)
    plt.tight_layout(); plt.show()

    scores = [dice_score(pred[k], masks[j]) for k, j in enumerate(picks)]
    print(f"true tumour px : {[int(masks[j].sum()) for j in picks]}")
    print(f"predicted   px : {[int(p.sum()) for p in pred]}")
    print(f"dice           : {[round(s, 2) for s in scores]}")
    print(f"\nmean Dice: {np.mean(scores):.3f}   across {len(set(patient_ids[picks]))} patients")
    print("\nSanity check: predicted pixel counts should be roughly comparable to the")
    print("true ones. If every slice reads 16384, the model is marking the whole image.")
except Exception as e:
    print("Pretrained demo unavailable:", repr(e)[:300])

Green is the radiologist. Red is the machine.

**In 10 hours, that red outline will be yours.**

---

## Exit ticket

Write the answers in the cell below. You will need them next session.

1. How many patients, how many slices?
2. What percentage of slices contain tumour?
3. What percentage of *pixels* are tumour?
4. How many institutions contributed data?
5. **Prediction:** if I built a model that just printed "no tumour anywhere" for every slice,
   what percentage of pixels would it get right?

Do not run any code for question 5. Just reason it out and write a number down.
We will find out next session whether you were right.

In [ ]:
#@markdown ### My dataset report
answer_1_patients_slices = ""  #@param {type:"string"}
answer_2_pct_slices_with_tumour = ""  #@param {type:"string"}
answer_3_pct_pixels_tumour = ""  #@param {type:"string"}
answer_4_institutions = ""  #@param {type:"string"}
answer_5_my_prediction = ""  #@param {type:"string"}
print("Saved. Bring these to Session 2.")